# Loading MAE model 

In [ ]:
import torch
from transformers import AutoImageProcessor, ViTMAEModel #the encoder backbone only
from PIL import Image
import requests

#ViTMAEForPreTraining. This is the FULL MAE training pipeline: masking, reconstruction, decoder, loss
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_name = "facebook/vit-mae-large"

processor = AutoImageProcessor.from_pretrained(model_name)
model = ViTMAEModel.from_pretrained(model_name).to(device) #Downloads pretrained weights.
model.eval()

#disable masking
model.config.mask_ratio = 0.0

# test image
url = "http://images.cocodataset.org/val2017/000000039769.jpg"
image = Image.open(requests.get(url, stream=True).raw).convert("RGB")

# preprocess
inputs = processor(images=image, return_tensors="pt").to(device)

with torch.no_grad():
    outputs = model(**inputs)

# inspecting shapes
last_hidden_state = outputs.last_hidden_state
print("last_hidden_state:", last_hidden_state.shape)  # should be (1, 197, 1024)

cls_embedding = last_hidden_state[:, 0, :] # didnt use it
mean_patch_embedding = last_hidden_state[:, 1:, :].mean(dim=1)

print("cls:", cls_embedding.shape)          # (1, 1024)
print("mean pooled:", mean_patch_embedding.shape)  # (1, 1024)

the following code is to show that outputs.last_hidden_state is the same as outputs.hidden_states[-1]

In [ ]:
# inspecting last layer
activations = {}

def hook_fn(name):
    def hook(module, input, output):
        activations[name] = output
    return hook

model.layernorm.register_forward_hook(
    hook_fn("final_layernorm")
)

outputs = model(
    **inputs,
    output_hidden_states=True
)

print(torch.allclose(
    outputs.last_hidden_state,
    activations["final_layernorm"]
))

In [ ]:
from transformers import AutoModel, AutoImageProcessor
from PIL import Image
import requests
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

model_name = "facebook/dinov2-large"

processor = AutoImageProcessor.from_pretrained(model_name)

model = AutoModel.from_pretrained(model_name)
model = model.to(device)
model.eval()

# test image
url = "http://images.cocodataset.org/val2017/000000039769.jpg"
image = Image.open(requests.get(url, stream=True).raw).convert("RGB")

# preprocess
inputs = processor(images=image, return_tensors="pt").to(device)

with torch.no_grad():
    outputs = model(**inputs)

print(outputs.keys())
print(outputs.last_hidden_state.shape)

# Dinov1 


In [ ]:
import torch

dino = torch.hub.load(
    "facebookresearch/dino:main",
    "dino_vitb16"
)

#print(hasattr(dino, "blocks"))
#print(len(dino.blocks))
#print(type(dino.blocks[0]))

hidden_states[0] is usually: embeddings before any transformer block.  
hidden_states[1] is after block 1,  
hidden_states[-1]: is after the last block before final LayerNorm.  
representations after each transformer block were extracted and the layer after final LayerNorm, resulting in 13 layers

In [ ]:
#testing if it has hidden states
import torch

# one test image
x = torch.randn(1, 3, 224, 224)

if torch.cuda.is_available():
    x = x.cuda()
    dino = dino.cuda()

features = {}

with torch.no_grad():
    tokens = dino.prepare_tokens(x)

    for i, block in enumerate(dino.blocks):
        tokens = block(tokens)

        pooled = tokens[:, 1:, :].mean(dim=1)

        features[f"layer_{i+1}"] = pooled

print(features.keys())

for k, v in features.items():
    print(k, v.shape)

# IBOT  



In [ ]:
import torch

ckpt = torch.load(
    "PATH",
    map_location="cpu"
)

print(type(ckpt))

if isinstance(ckpt, dict):
    print(ckpt.keys())

In [ ]:
sd = ckpt["state_dict"]

print(len(sd))

for i, k in enumerate(sd.keys()):
    print(k)
    if i >= 30:
        break

In [ ]:
from functools import partial
import torch.nn as nn


import sys

sys.path.append("PATH")

from models import vision_transformer

ibot_model = vision_transformer.VisionTransformer(
    patch_size=16,
    embed_dim=1024,
    depth=24,
    num_heads=16,
    mlp_ratio=4,
    qkv_bias=True,
    norm_layer=partial(nn.LayerNorm, eps=1e-6),
)

msg = ibot_model.load_state_dict(sd, strict=True)

print(msg)

# Feature extraction

In [ ]:
from net2brain.utils.download_datasets import DatasetBonnerPNAS2017, DatasetAlgonauts_NSD_Shared
from pprint import pprint
import os

paths_NSD_shared  = DatasetAlgonauts_NSD_Shared.load_dataset()
#pprint(paths_NSD_shared)
#paths_NSD_shared.keys()


stimuli_path = paths_NSD_shared["images"]

In [ ]:
from torchvision import transforms as trn
import torchextractor as tx
from net2brain.feature_extraction import FeatureExtractor
import torch


# preprocessing
def my_preprocessor(image, model_name, device):
    """
    Args:
        image (Union[Image.Image, List[Image.Image]]): A PIL Image or a list of PIL Images.
        model_name (str): The name of the model, used to determine specific preprocessing if necessary.
        device (str): The device to which the tensor should be transferred ('cuda' for GPU, 'cpu' for CPU).

    Returns:
        Union[torch.Tensor, List[torch.Tensor]]: The preprocessed image(s) as PyTorch tensor(s).
    """

    transforms = trn.Compose([
        trn.Resize((224, 224)),
        trn.ToTensor(),
        trn.Normalize(
            [0.485, 0.456, 0.406],
            [0.229, 0.224, 0.225]
        )
    ])

    img_tensor = transforms(image).unsqueeze(0)

    if device == 'cuda':
        img_tensor = img_tensor.cuda()

    return img_tensor


# generic CNN extractor

def my_extractor(preprocessed_data, layers_to_extract, model):

    extractor_model = tx.Extractor(model, layers_to_extract)

    _, features = extractor_model(preprocessed_data)

    return features


# generic transformer extractor
# (JEPA / MAE / DINOv2)

def my_transformer_extractor(preprocessed_data, layers_to_extract, model):

    outputs = model.model(
        preprocessed_data,
        output_hidden_states=True
    )

    hidden_states = outputs.hidden_states

    features = {}

    for i, layer in enumerate(hidden_states):

        # remove CLS token
        layer = layer[:, 1:, :]

        # mean-pool patch tokens
        layer = layer.mean(dim=1)

        features[f"layer_{i}"] = layer

    return features

def my_mae_transformer_extractor(preprocessed_data, layers_to_extract, model):
    outputs = model.model(
        preprocessed_data,
        output_hidden_states=True
    )

    hidden_states = outputs.hidden_states

    features = {}

    # hidden_states[0] = input embeddings before any transformer block
    # hidden_states[1] = output of transformer block 1
    # ...
    # hidden_states[24] = output of transformer block 24

    for block_num in range(1, len(hidden_states)):
        layer = hidden_states[block_num]

        # CLS token
        layer = layer[:, 0, :]

        features[f"layer_{block_num}"] = layer

    return features


# DINOv1
def my_dino_transformer_extractor(preprocessed_data, layers_to_extract, model):

    dino = model.model
    features = {}

    with torch.no_grad():
        tokens = dino.prepare_tokens(preprocessed_data)

        for i, block in enumerate(dino.blocks):
            tokens = block(tokens)

            # Use CLS token
            pooled = tokens[:, 0, :]

            if i == len(dino.blocks) - 1:
                features[f"layer_{i+1}_pre_norm"] = pooled
            else:
                features[f"layer_{i+1}"] = pooled

        # Apply final LayerNorm
        tokens = dino.norm(tokens)

        # Use CLS token after final LayerNorm
        pooled = tokens[:, 0, :]

        features[f"layer_{len(dino.blocks)}_post_norm"] = pooled

    return features


# IBOT
def my_ibot_transformer_extractor(preprocessed_data, layers_to_extract, model):

    ibot = model.model
    features = {}

    with torch.no_grad():
        tokens = ibot.prepare_tokens(preprocessed_data)

        for i, block in enumerate(ibot.blocks):
            tokens = block(tokens)

            # Use CLS token
            pooled = tokens[:, 0, :]

            if i == len(ibot.blocks) - 1:
                features[f"layer_{i+1}_pre_norm"] = pooled
            else:
                features[f"layer_{i+1}"] = pooled

        # Final LayerNorm
        tokens = ibot.norm(tokens)

        pooled = tokens[:, 0, :]

        features[f"layer_{len(ibot.blocks)}_post_norm"] = pooled

    return features

# cleaner

def my_cleaner(features):
    return features

In [ ]:
import torch.nn as nn

class TransformerFeatureExtractor(nn.Module):

    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, x):

        with torch.no_grad():
            outputs = self.model(pixel_values=x)

        return outputs
#transformer_extractor = TransformerFeatureExtractor(model)

class MAEFeatureExtractor(nn.Module):

    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, x):

        with torch.no_grad():
            outputs = self.model(pixel_values=x)

        return outputs
mae_wrapper = MAEFeatureExtractor(model)



class DINOFeatureExtractor(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

#dino_wrapper = DINOFeatureExtractor(dino)


#dino_wrapper = dino_wrapper.to(device)
#dino_wrapper.eval()
class IBOTFeatureExtractor(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

#ibot_wrapper = IBOTFeatureExtractor(ibot_model)

#ibot_wrapper = ibot_wrapper.to(device)
#ibot_wrapper.eval()

In [ ]:
# add MAE extractor. 
from net2brain.feature_extraction import FeatureExtractor
fx = FeatureExtractor(
    model=mae_wrapper,
    device='cuda' if torch.cuda.is_available() else 'cpu',
    preprocessor=my_preprocessor,
    feature_cleaner=my_cleaner,
    extraction_function=my_mae_transformer_extractor
)
fx.extract(
    data_path=stimuli_path,
    save_path="PATH",
    consolidate_per_layer=True
)

In [ ]:
# DINOv1 extractor
from net2brain.feature_extraction import FeatureExtractor
fx = FeatureExtractor(
    model=dino_wrapper,
    device='cuda' if torch.cuda.is_available() else 'cpu',
    preprocessor=my_preprocessor,
    feature_cleaner=my_cleaner,
    extraction_function=my_dino_transformer_extractor
)

fx.extract(
    data_path=stimuli_path,
    save_path="PA,
    consolidate_per_layer=False
)

In [ ]:
# ibot extractor
from net2brain.feature_extraction import FeatureExtractor
fx = FeatureExtractor(
    model=ibot_wrapper,
    device='cuda' if torch.cuda.is_available() else 'cpu',
    preprocessor=my_preprocessor,
    feature_cleaner=my_cleaner,
    extraction_function=my_ibot_transformer_extractor
)

fx.extract(
    data_path=stimuli_path,
    save_path=r'C:\Users\louai\Net2Brain\notebooks\0_FEATURE EXTRACTIONS\_unconsolidated\_correct\IBOT_Feat_nsd_an_cls_unconsolidated',
    consolidate_per_layer=False  # True: all images voxels will be included in each roi file -> 12 npy files. False: each img own 
)